# Bayesian Credibility Inference

Owner: Member 5

This notebook combines extraction confidence, entity-linking confidence, KG reasoning status, KG confidence, and optional LIAR speaker-history metadata into final credibility probabilities.

## Role in the Project

Notebook 4 produces `04_KG_Reasoning/kg_results.json`, where each claim is marked as `supported`, `contradicted`, or `unknown` by conservative KG rules. This notebook converts that symbolic result into `P(true)`, `P(false)`, and a final verdict for the downstream report and responsible AI reflection.

In [ ]:
from pathlib import Path
import json
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'LIAR_dataset').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

MODULE_DIR = PROJECT_ROOT / '05_Bayesian_Inference'
sys.path.insert(0, str(MODULE_DIR))

from run_bayesian_inference import run_bayesian_inference

INPUT_PATH = PROJECT_ROOT / '04_KG_Reasoning' / 'kg_results.json'
LINKED_INPUT_PATH = PROJECT_ROOT / '03_Entity_Linking_KR' / 'linked_entities.json'
OUTPUT_PATH = MODULE_DIR / 'final_verdicts.json'
SUMMARY_PATH = MODULE_DIR / 'final_verdict_summary.json'

INPUT_PATH, LINKED_INPUT_PATH, OUTPUT_PATH, SUMMARY_PATH

## Input Preview

The preferred input is the KG reasoning output from Notebook 4. The Bayesian runner also reads Notebook 3's linked-entity output when available so that extraction and linking confidence can be used to scale evidence strength.

In [ ]:
kg_records = json.loads(INPUT_PATH.read_text())
kg_records[:3]

## Bayesian Method

The model is intentionally conservative:

- start from a neutral `P(true)=0.50` prior
- adjust the prior only slightly with LIAR speaker-history counts when available
- treat KG support as positive evidence and KG contradiction as negative evidence
- scale KG evidence by extraction confidence, entity-linking confidence, and KG confidence
- keep `unknown` KG results near the prior instead of forcing every claim into true or false

In [ ]:
results = run_bayesian_inference(PROJECT_ROOT)
summary = results['summary']
summary

## Output Preview

Each output record contains the required final-verdict fields: `claim_id`, `probability_true`, `probability_false`, `final_verdict`, `decision_confidence`, and `reasoning`. Extra context is retained for traceability.

In [ ]:
final_verdicts = results['final_verdicts']
final_verdicts[:5]

## Verdict Distribution

This distribution supports the empirical analysis section of the report. A large `uncertain` count is expected when KG reasoning does not have enough evidence to support or contradict most claims.

In [ ]:
from collections import Counter

Counter(record['final_verdict'] for record in final_verdicts)

## High-Confidence Examples

The rows below are the strongest Bayesian decisions. They should be inspected in the report because high confidence depends on the upstream extraction, linking, and KG evidence quality.

In [ ]:
preview_fields = [
    'claim_id', 'final_verdict', 'probability_true', 'probability_false',
    'decision_confidence', 'kg_status', 'kg_confidence', 'reference_label', 'raw_claim'
]

[
    {field: record.get(field) for field in preview_fields}
    for record in sorted(final_verdicts, key=lambda item: item['decision_confidence'], reverse=True)[:10]
]

## Reference-Label Check

`reference_label` comes from the LIAR dataset and is used only for evaluation. It is not used to compute `P(true)` or the final verdict.

In [ ]:
summary.get('reference_bucket_alignment')

## Report Handoff

The runner writes `final_verdicts.json`, `final_verdict_summary.json`, and `report_assets/tables/bayesian_verdict_distribution.csv`. These files provide the final credibility results for the empirical analysis and conclusion sections.

In [ ]:
json.loads(SUMMARY_PATH.read_text())

## Limitations

The Bayesian output is only as reliable as the upstream evidence. Most KG results are `unknown` because the project uses conservative local reasoning rules, so many claims remain uncertain. Speaker-history priors are deliberately weak because they can encode political and historical bias. The model therefore favours transparent uncertainty over overconfident misinformation labels.